# Module 5: LangSmith Engine — Close the Loop Automatically

> *"Your proactive agent engineer."* — LangSmith

> Part of the **Modular Workshops** series. Standalone, ~30 min. **Capstone** — runs on the agent you deployed in Module 3.

In Module 4 you built the improvement loop **by hand**: trace → online eval scores a run → a run rule routes low scores to an annotation queue → a human reviews → fixes feed the next dataset.

**LangSmith Engine automates that entire loop** — and goes further. It reads your *deployed* agent's traces and runs five stages on its own:

1. **Detect** — find a *recurring* failure across many traces
2. **Diagnose** — trace the root cause into your connected GitHub source
3. **Propose a fix** — open a pull request (code or prompt)
4. **Deploy an evaluator** — add an online check so the issue can't silently regress
5. **Auto-reopen** — if the issue resurfaces after being resolved, reopen it

Engine's analysis runs automatically in the background — the first pass takes ~20 minutes, then it rescans every ~6h. You'll seed a few traces and turn Engine on, then walk through the issue it found, the fix it proposes, and the evaluator it deploys.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

This module runs on the agent you **deployed in Module 3** — Engine reads the deployed agent's production traces. Set its URL in `.env` as `LANGSMITH_DEPLOYMENT_URL` (or paste it into the seed cell below).

In [3]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os
from langsmith import Client

client = Client()
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
print("Project:", project_name)
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))

Project: modular-workshops
LANGSMITH_API_KEY set: True


### Seed a few date-demanding traces

This cell fires a handful of date-demanding queries at the **deployed** agent via the LangGraph SDK, giving Engine real traces to analyze (it needs only a couple to open an issue).

Each prompt asks for **recent, dated** information. The catch: `utils/search.py` formats Tavily hits with only title/url/content and **drops `published_date`** — so the agent answers "latest"/"recent" questions with **no source dates**. Engine flags that as a **hallucination** pattern: confident recency claims with nothing to ground them.

In [ ]:
from langgraph_sdk import get_client

deployment_url = os.environ.get("LANGSMITH_DEPLOYMENT_URL") or "<your-deployment-url>"
sdk = get_client(url=deployment_url, api_key=os.environ.get("LANGSMITH_API_KEY"))

# Answer inline so the run doesn't pause on the deep agent's write_file HITL interrupt.
SUFFIX = " Answer inline in your reply; do not write any files."
date_prompts = [
    "What are the latest AI agent frameworks released this year? Give the publication date of each source.",
    "What's the most recent LangChain release, and when was it published?",
    "Summarize the top AI research announcements from the last month, with dates.",
    "What changed in LangGraph most recently? Include when each change shipped.",
    "What is the newest model from Anthropic and what date was it announced?",
]

for q in date_prompts:
    await sdk.runs.wait(
        None,                 # stateless run (no thread)
        "deep_agent",         # graph name from langgraph.json
        input={"messages": [{"role": "user", "content": q + SUFFIX}]},
        metadata={"demo": "engine"},
    )
    print("seeded:", q)

print(f"\nSeeded {len(date_prompts)} traces into project: {project_name}")
print("Now turn Engine on below.")

## 1. Turn on Engine

With traces in the project, point Engine at the agent. In LangSmith:

1. Open the **Tracing Project** for this agent — the project your traces above landed in.
2. Click the **Engine** tab.
3. **Connect code repository** — connect this repo so Engine can diagnose from source and open a fix PR.
4. *(Optional)* **Connect context** — point Engine at any skills or memory files the agent uses (e.g. `agents/deep_agent/skills/`, `AGENTS.md`) so it reasons with the same context the agent has.
5. Under categories, select **Hallucinations** — the class our undated-answers defect falls under.
6. Start the analysis. The first pass takes ~20 min; after that Engine rescans every ~6h.

Once analysis finishes, the rest of this module walks through what it found.

## 2. Find the issue Engine filed

When the first analysis finishes, Engine's issues appear in the **Engine** tab. You can also list them from the CLI:

In [ ]:
!cd "{project_root}" && langsmith project issues list --project "{project_name}"

## 3. Review the issue

Open the issue for our defect (something like *Research answers omit source publication dates*). Each issue has a **title**, a **severity score**, a **category**, and a **description** — read down it the way you'd review a teammate's investigation:

- **Linked Traces** — the exact production traces Engine associated with the issue; these are the runs it used to locate the problem, so you can verify the pattern yourself.
- **Proposed Fix** — **Engine writes the fix** and shows the diff with an explanation: a **context fix** (update a prompt, skill, or memory file) and/or a **code fix** (update the code). Review it on its merits — it's generated from *your* source, so let the proposal speak for itself rather than assuming what it'll be. Open the code fix as a GitHub PR and read the diff like any other.
- **Suggested Evaluator** — an online check Engine recommends so the issue can't silently come back (you'll apply it next).

> The main flow **leaves the gap in place** so the issue persists across demos — don't merge yet. See *Close the loop* below to resolve it end to end.

## 4. Apply the suggested evaluator

A fix closes today's gap; an evaluator **prevents the issue from coming back**. The issue's **Suggested Evaluator** is an online check Engine wrote for exactly this pattern — apply it and every new trace is scored automatically, so the issue reopens on its own if the regression returns. It completes the **feedback loop from traces to fixes to evals**, and it's the same kind of online evaluator you built by hand in Module 4 with `create_run_rule(...)` — except Engine wrote it and tied it to the issue.

*Optional:* the cell below builds an equivalent check in code, if you want to see what that evaluator is doing under the hood.

In [ ]:
from utils.langsmith_rules import create_run_rule

date_judge_prompt = (
    "You score whether an assistant's answer to a time-sensitive question cites "
    "the publication dates of its sources. Reply with dates_cited (true/false) "
    "and one sentence of comment. Mark false if the answer makes 'latest'/'recent' "
    "claims without attributing any source dates."
)

date_judge_schema = {
    "title": "dates_cited",
    "description": "Whether time-sensitive answers cite source publication dates.",
    "type": "object",
    "properties": {
        "dates_cited": {"type": "boolean", "description": "True if source dates are cited"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["dates_cited", "comment"],
    "strict": True,
}

# Same helper as Module 4 -- the equivalent of the evaluator Engine suggests on the issue.
regression_rule = create_run_rule(
    client,
    project_name=project_name,
    display_name="engine-regression-dates-cited",
    sampling_rate=1.0,
    filter="eq(is_root, true)",
    llm_judge_prompt=date_judge_prompt,
    llm_judge_schema=date_judge_schema,
)
print("Regression evaluator rule ID:", regression_rule["id"])
print("Open in UI:", regression_rule["url"])

## 5. Close the loop (optional)

The main flow stops here: the gap stays in the code, so the issue persists and you can reuse it across demos. To watch Engine's loop close end to end instead:

1. **Merge** Engine's code-fix PR.
2. **Redeploy** the agent (Module 3, `langgraph deploy`).
3. **Re-run the seed cell.** The new answers carry source dates, the evaluator passes, and Engine **resolves** the issue on its next rescan.

Heads-up: this actually fixes the bug, so **to demo again you'll need to re-introduce it** — revert the merge (or drop `published_date` from the formatting in `utils/search.py`) and redeploy, and Engine re-detects it.

*Good to know:* Engine keeps **running automatically in the background** — it rescans every ~6h and auto-reopens regressions on its own, can emit webhooks (issue detected / resolved / reopened), and is metered in LCUs (1 LCU = $1.50) with an Org-Admin spend limit.

## Recap — Engine vs. the hand-built loop

| Engine stage | You did this by hand in Module 4 | Engine automates |
|---|---|---|
| **Detect** | Queried traces with `list_runs` + filters | Finds *recurring* failures across traces on its own |
| **Diagnose** | Read traces, guessed the cause | Traces the cause into connected GitHub source |
| **Propose fix** | Edited code yourself | Opens a PR (code or prompt) |
| **Add an evaluator** | `create_run_rule(...)` | Suggests an evaluator you apply in one click |
| **Route to human** | Annotation queue rule | Files issues for review + recommends offline examples |
| **Re-check** | Re-ran evals manually | Rescans every ~6h, auto-reopens regressions |

Same loop you built in Module 4 — now detected, diagnosed, fixed, and guarded automatically.

**Docs:** [LangSmith Engine](https://docs.langchain.com/langsmith/engine) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)